<center><h1>NLP - N-grams</center>

<b>Maestría</b>: Inteligencia Artificial Aplicada <br>
<b>Asignatura:</b> Procesamiento Acelerado de Lenguaje Natural <br>
<b>Profesor:</b> Edwin J. Rueda

## ¿Qué son los n-gramas?

Un $n$-grama es una secuencia de $n$ elementos (palabras, tokens, caracteres, etc) consecutivos en un texto. Básicamente consiste en crear secuencias de tamaño $n$ con desplazamiento de una posición.

<table><thead>
  <tr>
    <th>n</th>
    <th class="tg-0lax">Type</th>
    <th class="tg-0lax">Input text: "un ejemplo de generacion de n-gramas"</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-baqh">1</td>
    <td class="tg-0lax">1-grama</td>
    <td class="tg-0lax">["un", "ejemplo", "de", "generacion", "de", "n-gramas"]</td>
  </tr>
  <tr>
    <td class="tg-baqh">2</td>
    <td class="tg-0lax">2-grama</td>
    <td class="tg-0lax">["un ejemplo", "ejemplo de", "de generacion", "generación de, "de n-gramas"]</td>
  </tr>
  <tr>
    <td class="tg-baqh">3</td>
    <td class="tg-0lax">3-grama</td>
    <td class="tg-0lax">["un ejemplo de",&nbsp;&nbsp;"ejemplo de generacion", "de generacion de", "generacion de n-gramas"]</td>
  </tr>
</tbody>
</table>


### ¿Por qué son útiles?

Los $n$-gramas nos permiten capturar patrones dentro de un texto. Lo cual es muy útil para tareas como:
- Clasificación de texto
- Predicción de la siguiente palabra (generador)
- Autocompletado
- Detección de plagio
- Reconocimiento de habla
- Correción de texto

##### Implementación básica en python
Para construir un `n-grama` a partir de una sentencía, debemos hacer lo siguiente:
- Iterar sobre la ventana de tokens `tokens[i:i+n]`
    - Donde `n` representa el tamaño de cada grama
    - `i` es el índice para iterar sobre todo el texto

In [4]:
def get_n_grams(sentence, n=1):
    words = sentence.lower().split()
    n_grams = []
    for i in range(len(words) - n + 1):
        ngram = tuple(words[i:i+n])
        n_grams.append(ngram)
    return n_grams

sentence = "este es un ejemplo de ngramas, y este es otro ejemplo"
get_n_grams(sentence, 3)

[('este', 'es', 'un'),
 ('es', 'un', 'ejemplo'),
 ('un', 'ejemplo', 'de'),
 ('ejemplo', 'de', 'ngramas')]

##### $n$-gramas usando nltk

In [5]:
from nltk.util import ngrams

ngrams = list(ngrams(sentence.split(),2))

print(ngrams)

[('este', 'es'), ('es', 'un'), ('un', 'ejemplo'), ('ejemplo', 'de'), ('de', 'ngramas')]


### Creación de un modelo lingüistico

Un ejemplo de uso de los `n-gramas` es para recomendar la palabra siguiente mas probable en una cadena de texto. Por ejemplo:

$$ P(W | S) = P(\text{feliz} | \text{yo estoy} ) $$

Para ello, tenemos que procesar el texto de entrenamiento en tokens, y posteriormente construir el conteo de secuencias de `n-gramas` seguido de una palabra `w`.

In [7]:
import numpy as np
import pandas as pd
from collections import defaultdict

def get_gram_count_matrix(corpus, n=3):
    """
    Creates a count matrix from the input corpus in a single pass through the corpus.
    
    Args:
        corpus: Pre-processed and tokenized corpus.
        n: n-grams
    
    Returns:
        subgrama: list of all (n-1) grams prefixes, row index
        vocabulary: list of all found words, the column index
        count_matrix: pandas dataframe with subgrama prefixes as rows, 
                      vocabulary words as columns 
                      and the counts of the subgrama/word combinations (i.e. trigrams) as values
    """
    subgrams = []
    vocabulary = []
    count_matrix_dict = defaultdict(dict)
    
    for i in range(len(corpus) - n + 1):
        grama = tuple(corpus[i : i + n])
        subgrama = grama[0: -1]
        last_word = grama[-1]
        if not subgrama in subgrams:
            subgrams.append(subgrama)
        if not last_word in vocabulary:
            vocabulary.append(last_word)
            
        if (subgrama, last_word) not in count_matrix_dict:
            count_matrix_dict[subgrama, last_word] = 0
        
        count_matrix_dict[subgrama, last_word] += 1
            
    count_matrix = np.zeros((len(subgrams), len(vocabulary)))
    for grama_key, grama_count in count_matrix_dict.items():
        count_matrix[subgrams.index(grama_key[0]), vocabulary.index(grama_key[1])] = grama_count
        
    count_matrix = pd.DataFrame(count_matrix, index=subgrams, columns=vocabulary)
    return subgrams, vocabulary, count_matrix

corpus = ['yo', 'estoy', 'feliz', 'porque', 'yo', 'estoy', 'aprendiendo', 'NLP', '.']

bigrams, vocabulary, count_matrix = get_gram_count_matrix(corpus, 3)

print(count_matrix)

                      feliz  porque   yo  estoy  aprendiendo  NLP    .
(yo, estoy)             1.0     0.0  0.0    0.0          1.0  0.0  0.0
(estoy, feliz)          0.0     1.0  0.0    0.0          0.0  0.0  0.0
(feliz, porque)         0.0     0.0  1.0    0.0          0.0  0.0  0.0
(porque, yo)            0.0     0.0  0.0    1.0          0.0  0.0  0.0
(estoy, aprendiendo)    0.0     0.0  0.0    0.0          0.0  1.0  0.0
(aprendiendo, NLP)      0.0     0.0  0.0    0.0          0.0  0.0  1.0


Con la matriz de conteo generada (note que la función `get_gram_count_matrix` nos retorna es un `DataFrame`), podemos calcular la probabilidad de que dado un `(n-1)grama`, siga una palabra `w`:

In [8]:
row_sums = count_matrix.sum(axis=1)
prob_matrix = count_matrix.div(row_sums, axis=0)
print("matriz de probabilidad:")
print(prob_matrix)

matriz de probabilidad:
                      feliz  porque   yo  estoy  aprendiendo  NLP    .
(yo, estoy)             0.5     0.0  0.0    0.0          0.5  0.0  0.0
(estoy, feliz)          0.0     1.0  0.0    0.0          0.0  0.0  0.0
(feliz, porque)         0.0     0.0  1.0    0.0          0.0  0.0  0.0
(porque, yo)            0.0     0.0  0.0    1.0          0.0  0.0  0.0
(estoy, aprendiendo)    0.0     0.0  0.0    0.0          0.0  1.0  0.0
(aprendiendo, NLP)      0.0     0.0  0.0    0.0          0.0  0.0  1.0


Tomamos el corpus `cess_esp` como ejemplo:

In [9]:
from nltk.corpus import cess_esp

init_words = cess_esp.words()[:20000]
len(init_words)

20000

In [10]:
import re

def preprocessing(tokens):
    new_tokens = []
    for token in tokens:
        new_tokens.append(re.sub(r"[^a-zA-Z0-9.?! ]+", "",token.lower()))
    return new_tokens

In [11]:
process_token = preprocessing(init_words)
print(init_words[:6])
print(process_token[:6])

['El', 'grupo', 'estatal', 'Electricité_de_France', '-Fpa-', 'EDF']
['el', 'grupo', 'estatal', 'electricitdefrance', 'fpa', 'edf']


computamos la matriz de conteo:

In [12]:
pre_grams_cess, vocabulary_cess, count_matrix_cess = get_gram_count_matrix(process_token, n=3)
print("Tamaño del vocabulario:", len(vocabulary_cess))

Tamaño del vocabulario: 4637


In [13]:
count_matrix_cess

,estatal,electricitdefrance,fpa,edf,fpt,anunci,hoy,,jueves,la,...,chatarra,enano,calificativos,entretenido,exigen,envezde,insultarse,profundicen,encasode,analistas
"(el, grupo)",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(grupo, estatal)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(estatal, electricitdefrance)",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(electricitdefrance, fpa)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(fpa, edf)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"(0, profundicen)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(profundicen, en)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(sus, programas)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(gobierno, encasode)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Computamos la matriz de probabilidad:

In [15]:
row_sums_cess = count_matrix_cess.sum(axis=1)
prob_matrix_cess = count_matrix_cess.div(row_sums_cess, axis=0)
print("matriz de probabilidad:")
print(prob_matrix_cess.head())

matriz de probabilidad:
                               estatal  electricitdefrance  fpa  edf  fpt  \
(el, grupo)                        0.5                 0.0  0.0  0.0  0.0   
(grupo, estatal)                   0.0                 1.0  0.0  0.0  0.0   
(estatal, electricitdefrance)      0.0                 0.0  1.0  0.0  0.0   
(electricitdefrance, fpa)          0.0                 0.0  0.0  1.0  0.0   
(fpa, edf)                         0.0                 0.0  0.0  0.0  1.0   

                               anunci  hoy       jueves   la  ...  chatarra  \
(el, grupo)                       0.0  0.0  0.0     0.0  0.0  ...       0.0   
(grupo, estatal)                  0.0  0.0  0.0     0.0  0.0  ...       0.0   
(estatal, electricitdefrance)     0.0  0.0  0.0     0.0  0.0  ...       0.0   
(electricitdefrance, fpa)         0.0  0.0  0.0     0.0  0.0  ...       0.0   
(fpa, edf)                        0.0  0.0  0.0     0.0  0.0  ...       0.0   

                               enano  

In [16]:
prob_matrix_cess.index = pd.MultiIndex.from_tuples(
    prob_matrix_cess.index,
    names=['w_prev', 'w_next']
)
prob_matrix_cess.head()

,,estatal,electricitdefrance,fpa,edf,fpt,anunci,hoy,,jueves,la,...,chatarra,enano,calificativos,entretenido,exigen,envezde,insultarse,profundicen,encasode,analistas
w_prev,w_next,,,,,,,,,,,,,,,,,,,,,
el,grupo,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
grupo,estatal,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
estatal,electricitdefrance,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
electricitdefrance,fpa,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
fpa,edf,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Asi, dada una nueva oración, podríamos obtener la palabra siguiente mas recomendada:

In [17]:
prefix = "el presidente".split()
print(prefix)

['el', 'presidente']


buscamos en el indice las plabras sugeridas:

In [19]:
words = prob_matrix_cess.loc[(prefix[0], prefix[1])]
candidate_words = words[words != 0]
print("palabras candidatas: ")
print(candidate_words)

palabras candidatas: 
del               0.25
de                0.45
en                0.05
cubano            0.05
estadounidense    0.05
chileno           0.05
hugochvez         0.05
venezolano        0.05
Name: (el, presidente), dtype: float64


In [22]:
print("Palabra sugerida: ")
candidate_words.index[np.argmax(candidate_words)]

Palabra sugerida: 


'de'

In [ ]:
blsa -> bolsa   (cambiar, agregar, eliminar)

#### Conclusiones

- Tenga en cuenta que la generación de $n$-gramas es un paso previo para la construcción de datasets para determinada tarea. En este caso, cada $n$-grama se convierte en un $feature$ para la entrada de un modelo de IA.
- Los $n$-gramas si bien capturan contexto, no generalizan del todo. Esto debido a que se necesitaría un $n$ muy grande para tener un mejor contexto, pero esto haría que se torne muy específico, creando representaciones dispersas.
-  Veremos como se combinan con representaciones vectoriales.